# Day 1 — Dataset access + controlled subset manifest

**Goal:** authenticate to PhysioNet, download *small* metadata/report files, build a reproducible subset manifest (2000 studies).

In [ ]:
!pip -q install pandas tqdm

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Download small files only (PhysioNet)

This downloads only metadata + reports archive (no TB-scale folders).

In [ ]:

USER = "YOUR_PHYSIONET_USERNAME"  # <-- change
# NOTE: You will be prompted for your PhysioNet password in the cell output.
!wget -N --user {USER} --ask-password -P {RAW} https://physionet.org/files/mimic-cxr/2.1.0/cxr-record-list.csv.gz
!wget -N --user {USER} --ask-password -P {RAW} https://physionet.org/files/mimic-cxr/2.1.0/cxr-study-list.csv.gz
!wget -N --user {USER} --ask-password -P {RAW} https://physionet.org/files/mimic-cxr/2.1.0/mimic-cxr-reports.zip

!ls -lh {RAW}


## 2) Load record list and create a reproducible subset of studies

In [ ]:

rec = pd.read_csv(f"{RAW}/cxr-record-list.csv.gz")
print("rec shape:", rec.shape)
print(rec.columns)

# Sample 2000 unique studies reproducibly
rng = np.random.default_rng(42)
study_ids = rec["study_id"].dropna().unique()
sampled_studies = rng.choice(study_ids, size=2000, replace=False)

subset = rec[rec["study_id"].isin(sampled_studies)].copy()
subset.to_csv(f"{BASE}/subset_2000_studies_manifest.csv", index=False)

print("Saved:", f"{BASE}/subset_2000_studies_manifest.csv")
print("Rows (dicoms):", len(subset))
print("Unique studies:", subset["study_id"].nunique())
print("Unique subjects:", subset["subject_id"].nunique())
subset.head()


## 3) Extract reports for sampled studies

In [ ]:

import zipfile, pathlib

zip_path = f"{RAW}/mimic-cxr-reports.zip"
assert os.path.exists(zip_path), f"Missing {zip_path}"

needed = set(map(int, subset["study_id"].unique()))
print("Needed reports:", len(needed))

extracted = 0
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        # Report files are typically like files/pXX/pXXXXXXX/sXXXXXXXX.txt
        if not name.endswith(".txt"): 
            continue
        # parse study id from filename s########.txt
        m = re.search(r"/s(\d+)\.txt$", name)
        if not m:
            continue
        sid = int(m.group(1))
        if sid in needed:
            z.extract(name, REPORTS_DIR)
            extracted += 1

print("Extracted reports:", extracted)
print("Reports saved under:", REPORTS_DIR)


## 4) Build a study_id → report_text table and merge into manifest

In [ ]:

# Gather extracted report paths
txt_paths = []
for root, _, files in os.walk(REPORTS_DIR):
    for fn in files:
        if fn.endswith(".txt") and fn.startswith("s"):
            txt_paths.append(os.path.join(root, fn))

print("Total .txt reports found:", len(txt_paths))
print("First 5:", [os.path.basename(p) for p in txt_paths[:5]])

# Map: study_id -> report_text
rows = []
for p in tqdm(txt_paths):
    m = re.match(r"s(\d+)\.txt$", os.path.basename(p))
    if not m:
        continue
    sid = int(m.group(1))
    if sid not in needed:
        continue
    with open(p, "r", errors="ignore") as f:
        rows.append({"study_id": sid, "report_text": f.read()})

reports_df = pd.DataFrame(rows).drop_duplicates("study_id")
reports_df.to_csv(f"{BASE}/reports_2000.csv", index=False)
print("Saved:", f"{BASE}/reports_2000.csv", "Rows:", len(reports_df))

merged = subset.merge(reports_df, on="study_id", how="left")
print("Manifest rows:", len(subset))
print("Merged rows:", len(merged))
print("Missing reports:", merged["report_text"].isna().sum())

merged.to_csv(f"{BASE}/manifest_with_reports_2000studies.csv", index=False)
print("Saved:", f"{BASE}/manifest_with_reports_2000studies.csv")
merged.head()


✅ **Day 1 deliverables:**
- `subset_2000_studies_manifest.csv`
- `reports_2000.csv`
- `manifest_with_reports_2000studies.csv`
- extracted report `.txt` files under `dataset/reports/`